# Narmada Basin Watch
## Automated Vegetation & Water Body Monitoring — Sentinel-2 + GEE

**Author:** Uzzal Saha
**Data:** Sentinel-2 SR Harmonized (Copernicus / ESA)  
**Platform:** Google Colab + Earth Engine Python API

Narmada Basin Watch
Automated Vegetation & Water Body Monitoring — Sentinel-2 + GEE

**Study Area:** Narmada River Basin, India (~98,796 km²)
**Data Source:** Sentinel-2 SR Harmonized (10m, 5-day revisit)
**Analysis Period:** January 2023 – December 2024

Pipeline Overview
1. Cloud-masked Sentinel-2 preprocessing
2. Spectral index computation (NDVI, NDWI, NDBI)
3. Monthly time-series extraction
4. Interactive multi-layer map
5. Basin-level statistics & vegetation class distribution

In [ ]:
!pip install -q earthengine-api geemap folium plotly geopandas rasterio matplotlib --quiet

In [ ]:
import ee
import geemap
import folium
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print("Imports successful.")

In [ ]:
ee.Authenticate()

PROJECT_ID = 'ee-uzzalsaha2026'
ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine authenticated and initialised.")
print(f"   Project: {PROJECT_ID}")

In [ ]:
# Narmada basin bounding box (approximate full basin extent)
BASIN_BOUNDS = [72.5, 21.2, 81.8, 23.8]   # [west, south, east, north]
narmada_aoi  = ee.Geometry.Rectangle(BASIN_BOUNDS)

# Basin centre (for map initialisation)
BASIN_CENTER = [22.5, 77.15]   # [lat, lon]

# Sub-region: Upper basin (Amarkantak origin area, MP)
upper_basin = ee.Geometry.Rectangle([80.5, 22.5, 81.8, 23.5])

# Sub-region: Lower basin (Gujarat end, Sardar Sarovar area)
lower_basin = ee.Geometry.Rectangle([72.5, 21.5, 74.5, 22.5])

print("Study area defined:")
print(f"   Full basin  : {BASIN_BOUNDS}")
print(f"   Center point: {BASIN_CENTER}")

In [ ]:
def mask_s2_clouds(image):
    """
    Cloud-mask Sentinel-2 SR using the Scene Classification Layer (SCL).
    Removes: cloud shadows (3), medium clouds (8), high clouds (9),
             cirrus (10), snow/ice (11).
    Keeps:   vegetation (4), bare soil (5), water (6), unclassified (7).
    """
    scl = image.select('SCL')
    cloud_mask = (scl.neq(3)   # not cloud shadow
                    .And(scl.neq(8))   # not medium probability cloud
                    .And(scl.neq(9))   # not high probability cloud
                    .And(scl.neq(10))  # not thin cirrus
                    .And(scl.neq(11))) # not snow/ice
    return image.updateMask(cloud_mask)


def compute_spectral_indices(image):
    """
    Add NDVI, NDWI, and NDBI bands to a Sentinel-2 image.

    NDVI  = (B8 - B4)  / (B8 + B4)   → Vegetation health  [-1, 1]
    NDWI  = (B3 - B8)  / (B3 + B8)   → Surface water      [-1, 1]
    NDBI  = (B11 - B8) / (B11 + B8)  → Built-up index     [-1, 1]
    """
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return image.addBands([ndvi, ndwi, ndbi])


# ── Load Sentinel-2 L2A Surface Reflectance collection ──
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(narmada_aoi)
        .filterDate('2023-01-01', '2024-12-31')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
        .map(compute_spectral_indices))

count = s2.size().getInfo()
print(f"Sentinel-2 collection loaded.")
print(f"   Images found (cloud < 20%, 2023–2024): {count}")

# Median annual composite for visualisation
composite_2023 = (s2.filterDate('2023-01-01', '2023-12-31').median())
composite_2024 = (s2.filterDate('2024-01-01', '2024-12-31').median())

print("   Annual median composites created for 2023 and 2024.")

In [ ]:
# Extract basin-wide mean NDVI for each month (2023 and 2024)

MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

def extract_monthly_mean(year, month):
    """Return mean NDVI and NDWI for given year/month over full basin."""
    m_start = f'{year}-{month:02d}-01'
    m_end   = f'{year}-{month:02d}-28'  # conservative end — GEE filters inclusively
    monthly_img = (s2.filterDate(m_start, m_end)
                     .mean()
                     .select(['NDVI', 'NDWI']))
    stats = monthly_img.reduceRegion(
        reducer   = ee.Reducer.mean(),
        geometry  = narmada_aoi,
        scale     = 2000,    # 2 km scale → fast, sufficient for basin-level stats
        maxPixels = 1e9,
        bestEffort= True
    )
    return stats.getInfo()


print(" Extracting monthly time series...")

records = []
for year in [2023, 2024]:
    for month in range(1, 13):
        try:
            result = extract_monthly_mean(year, month)
            ndvi_val = result.get('NDVI')
            ndwi_val = result.get('NDWI')
            records.append({
                'year'      : year,
                'month'     : month,
                'month_name': MONTH_NAMES[month - 1],
                'period'    : f"{MONTH_NAMES[month-1]} {year}",
                'ndvi'      : round(ndvi_val, 4) if ndvi_val is not None else None,
                'ndwi'      : round(ndwi_val, 4) if ndwi_val is not None else None
            })
        except Exception as e:
            print(f"  ⚠ Skipped {MONTH_NAMES[month-1]} {year}: {e}")

df = pd.DataFrame(records).dropna(subset=['ndvi'])
print(f"\n Time series extracted: {len(df)} months with valid data.")
print(df[['period', 'ndvi', 'ndwi']].to_string(index=False))

In [ ]:
df_2023 = df[df['year'] == 2023].reset_index(drop=True)
df_2024 = df[df['year'] == 2024].reset_index(drop=True)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        'Monthly Mean NDVI — Narmada Basin (Vegetation Health)',
        'Monthly Mean NDWI — Narmada Basin (Surface Water)'
    ],
    vertical_spacing=0.15,
    shared_xaxes=False
)

# ── NDVI panel ──
for yr, clr, dash in [(2023, '#2ECC71', 'solid'), (2024, '#27AE60', 'dot')]:
    d = df[df['year'] == yr]
    fig.add_trace(go.Scatter(
        x=d['month_name'], y=d['ndvi'],
        mode='lines+markers',
        name=f'NDVI {yr}',
        line=dict(color=clr, width=2.5, dash=dash),
        marker=dict(size=7),
        fill='tozeroy' if yr == 2023 else None,
        fillcolor='rgba(46,204,113,0.08)'
    ), row=1, col=1)

# Vegetation threshold reference lines
fig.add_hline(y=0.5,  line_dash='dash', line_color='darkgreen',
              annotation_text='Dense vegetation (NDVI > 0.5)',
              annotation_position='top right', row=1, col=1)
fig.add_hline(y=0.2,  line_dash='dash', line_color='orange',
              annotation_text='Sparse vegetation (NDVI > 0.2)',
              annotation_position='bottom right', row=1, col=1)

# ── NDWI panel ──
for yr, clr, dash in [(2023, '#3498DB', 'solid'), (2024, '#2980B9', 'dot')]:
    d = df[df['year'] == yr]
    fig.add_trace(go.Scatter(
        x=d['month_name'], y=d['ndwi'],
        mode='lines+markers',
        name=f'NDWI {yr}',
        line=dict(color=clr, width=2.5, dash=dash),
        marker=dict(size=7)
    ), row=2, col=1)

fig.add_hline(y=0.0, line_dash='dash', line_color='steelblue',
              annotation_text='Water presence threshold (NDWI > 0)',
              row=2, col=1)

fig.update_layout(
    title=dict(text='🛰️ Narmada River Basin — Vegetation & Water Monitor (Sentinel-2)',
               font=dict(size=16)),
    height=700,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    font=dict(family='Arial', size=12)
)
fig.update_yaxes(title_text='NDVI',  range=[-0.1, 0.9], row=1, col=1)
fig.update_yaxes(title_text='NDWI',  range=[-0.6, 0.6], row=2, col=1)
fig.update_xaxes(title_text='Month', row=2, col=1)

fig.show()
fig.write_html('ndvi_ndwi_timeseries.html')
print(" Time series chart saved as 'ndvi_ndwi_timeseries.html'")

In [ ]:
# Classify pixels by NDVI and compute area per class

NDVI_CLASSES = {
    'Water / Bare soil (NDVI < 0.1)':         {'min': -1.0, 'max': 0.1,  'color': '#A0522D'},
    'Sparse vegetation (0.1 – 0.3)':           {'min':  0.1, 'max': 0.3,  'color': '#F4D03F'},
    'Moderate vegetation (0.3 – 0.5)':         {'min':  0.3, 'max': 0.5,  'color': '#82E0AA'},
    'Dense / Healthy vegetation (NDVI > 0.5)': {'min':  0.5, 'max':  1.0, 'color': '#1D6A4E'},
}

print("Computing vegetation class distribution...")

ndvi_2023 = composite_2023.select('NDVI')
class_areas = {}

for class_name, params in NDVI_CLASSES.items():
    class_mask   = ndvi_2023.gte(params['min']).And(ndvi_2023.lt(params['max']))
    pixel_count  = class_mask.reduceRegion(
        reducer   = ee.Reducer.sum(),
        geometry  = narmada_aoi,
        scale     = 1000,
        maxPixels = 1e9,
        bestEffort= True
    ).getInfo()
    n_pixels = pixel_count.get('NDVI', 0)
    area_km2 = (n_pixels * 1000 * 1000) / 1e6  # pixels at 1km scale → km²
    class_areas[class_name] = round(area_km2, 1)

df_classes = pd.DataFrame({
    'Class' : list(class_areas.keys()),
    'Area_km2': list(class_areas.values()),
    'Color'   : [v['color'] for v in NDVI_CLASSES.values()]
})
df_classes['Percent'] = (df_classes['Area_km2'] / df_classes['Area_km2'].sum() * 100).round(1)

print("\n=== Vegetation Class Distribution — Narmada Basin (2023) ===")
print(df_classes[['Class', 'Area_km2', 'Percent']].to_string(index=False))

fig_pie = go.Figure(go.Pie(
    labels  = df_classes['Class'],
    values  = df_classes['Area_km2'],
    marker  = dict(colors=df_classes['Color'].tolist()),
    textinfo= 'label+percent',
    hole    = 0.35,
))
fig_pie.update_layout(
    title=dict(text='Vegetation Class Distribution — Narmada Basin (2023)',
               font=dict(size=14)),
    height=450,
    showlegend=False,
    font=dict(family='Arial', size=11)
)
fig_pie.show()

In [ ]:
# Visualisation parameters
VIS_TRUE_COLOUR = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4}

VIS_NDVI = {
    'min': -0.2, 'max': 0.85,
    'palette': ['#8B4513', '#D2691E', '#F5DEB3',  # bare / dry
                '#ADFF2F', '#90EE90',               # sparse
                '#32CD32', '#228B22', '#006400']    # dense forest
}

VIS_NDWI = {
    'min': -0.4, 'max': 0.4,
    'palette': ['#FFFFE0', '#87CEEB', '#4169E1', '#00008B', '#00003B']
}

VIS_NDBI = {
    'min': -0.3, 'max': 0.3,
    'palette': ['#FFFFFF', '#FFD700', '#FF8C00', '#FF4500', '#8B0000']
}

# Build geemap interactive map
Map = geemap.Map(center=BASIN_CENTER, zoom=7)
Map.add_basemap('HYBRID')   # Satellite + labels background

# Add layers
Map.addLayer(composite_2023.select(['B4','B3','B2']),
             VIS_TRUE_COLOUR, 'True Colour (2023)', shown=False)
Map.addLayer(composite_2023.select('NDVI'),
             VIS_NDVI, 'NDVI — Vegetation (2023)', shown=True)
Map.addLayer(composite_2023.select('NDWI'),
             VIS_NDWI, 'NDWI — Water Bodies (2023)', shown=False)
Map.addLayer(composite_2023.select('NDBI'),
             VIS_NDBI, 'NDBI — Built-up (2023)', shown=False)

# Highlight NDWI change: 2023 vs 2024 water bodies
ndwi_change = (composite_2024.select('NDWI')
               .subtract(composite_2023.select('NDWI')))
Map.addLayer(ndwi_change,
             {'min': -0.3, 'max': 0.3, 'palette': ['red','white','blue']},
             'NDWI Change 2023→2024', shown=False)

# Basin boundary outline
basin_outline = (ee.Image().byte()
                 .paint(featureCollection=ee.FeatureCollection([ee.Feature(narmada_aoi)]),
                        color=1, width=2))
Map.addLayer(basin_outline, {'palette': ['#FF4500']}, 'Basin Boundary')

# Add colourbar legend for NDVI
Map.add_colorbar(
    vis_params=VIS_NDVI,
    label='NDVI',
    orientation='horizontal',
    position='bottomright',
    transparent_bg=True
)

Map.addLayerControl()
print("Interactive map created. Use the layer panel (top-right) to toggle layers.")
display(Map)

In [ ]:
print("Computing basin-level statistics...")

def get_band_stats(image, band_name, geometry, scale=2000):
    """Return mean, min, max, stdDev for a band over a geometry."""
    stats = image.select(band_name).reduceRegion(
        reducer=(ee.Reducer.mean()
                   .combine(ee.Reducer.min(),    sharedInputs=True)
                   .combine(ee.Reducer.max(),    sharedInputs=True)
                   .combine(ee.Reducer.stdDev(), sharedInputs=True)),
        geometry  = geometry,
        scale     = scale,
        maxPixels = 1e9,
        bestEffort= True
    ).getInfo()
    return {
        'mean'  : round(stats.get(f'{band_name}_mean',   0), 4),
        'min'   : round(stats.get(f'{band_name}_min',    0), 4),
        'max'   : round(stats.get(f'{band_name}_max',    0), 4),
        'stddev': round(stats.get(f'{band_name}_stdDev', 0), 4),
    }

stats_rows = []
for band, label in [('NDVI', 'Vegetation (NDVI)'),
                    ('NDWI', 'Surface Water (NDWI)'),
                    ('NDBI', 'Built-up (NDBI)')]:
    s_2023 = get_band_stats(composite_2023, band, narmada_aoi)
    s_2024 = get_band_stats(composite_2024, band, narmada_aoi)
    stats_rows.append({
        'Index'          : label,
        'Mean 2023'      : s_2023['mean'],
        'Mean 2024'      : s_2024['mean'],
        'Δ Mean'         : round(s_2024['mean'] - s_2023['mean'], 4),
        'Std Dev (2023)' : s_2023['stddev'],
        'Max (2023)'     : s_2023['max'],
        'Min (2023)'     : s_2023['min'],
    })

df_stats = pd.DataFrame(stats_rows)

print("\n" + "="*65)
print("  Narmada Basin — Spectral Index Statistics (Sentinel-2)")
print("="*65)
print(df_stats.to_string(index=False))
print("="*65)
print("\n Interpretation:")
print("   NDVI > 0.5  → Dense healthy vegetation (forests, active cropland)")
print("   NDVI 0.2–0.5 → Sparse / moderate vegetation")
print("   NDWI > 0.0  → Surface water presence")
print("   NDBI > 0.0  → Urban / built-up surfaces")
print("\n   Positive Δ Mean in NDVI → vegetation improved 2023 → 2024")
print("   Positive Δ Mean in NDWI → water body extent increased")

# Export stats to CSV
df_stats.to_csv('narmada_basin_statistics.csv', index=False)
print("\n Statistics saved to 'narmada_basin_statistics.csv'")

In [ ]:
from google.colab import files

# For saving the interactive map as HTML
try:
    Map.to_html('narmada_interactive_map.html')
    print(" Interactive map saved as 'narmada_interactive_map.html'")
except Exception as e:
    print(f"ℹ Map HTML export: {e} — map is still viewable in Colab above.")

# Downloading files to local machine
print("\n Downloading output files...")
try:
    files.download('ndvi_ndwi_timeseries.html')
    files.download('narmada_basin_statistics.csv')
except:
    print("ℹ Auto-download skipped (not in interactive Colab session).")

print("\n" + "="*55)
print("  Analysis Complete!")
print("="*55)
print("  Outputs generated:")
print("  • ndvi_ndwi_timeseries.html   — Interactive Plotly chart")
print("  • narmada_interactive_map.html — Geemap layer map")
print("  • narmada_basin_statistics.csv — Summary statistics")
print("="*55)